In [45]:
import pandas as pd
import os

In [46]:
# Load data
df = pd.read_csv('../data/raw/car_data.csv')

print(f'DataFrame Shape: {df.shape}')
print(f'Missing value: {df.isnull().sum()}')

DataFrame Shape: (301, 9)
Missing value: Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64


In [47]:
# Drop duplicated rows
print(df[df.duplicated(keep=False)])
print('................................................................')
print(f'Before dropping: {df.shape}')

df.drop_duplicates(inplace=True)
print(f'After dropping: {df.shape}')
print('................................................................')

    Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
15    ertiga  2016           7.75          10.79       43000    Diesel   
17    ertiga  2016           7.75          10.79       43000    Diesel   
51  fortuner  2015          23.00          30.61       40000    Diesel   
93  fortuner  2015          23.00          30.61       40000    Diesel   

   Selling_type Transmission  Owner  
15       Dealer       Manual      0  
17       Dealer       Manual      0  
51       Dealer    Automatic      0  
93       Dealer    Automatic      0  
................................................................
Before dropping: (301, 9)
After dropping: (299, 9)
................................................................


In [48]:
print(df[df['Driven_kms'] == 500000])
print(df[df['Car_Name'] == 'Activa 3g'])

# Drop row with 500000 driven_kms
df = df[df['Driven_kms'] != 500000]

      Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
196  Activa 3g  2008           0.17           0.52      500000    Petrol   

    Selling_type Transmission  Owner  
196   Individual    Automatic      0  
      Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
165  Activa 3g  2016           0.45           0.54         500    Petrol   
196  Activa 3g  2008           0.17           0.52      500000    Petrol   

    Selling_type Transmission  Owner  
165   Individual    Automatic      0  
196   Individual    Automatic      0  


### Driven_kms Outlier Investigation

The dataset contains one vehicle with a mileage of 500,000 km. This vehicle is a 2008 Activa 3G with a selling price of 0.17 lakh and a present price of 0.52 lakh, corresponding to a depreciation rate of approximately 67.3%. While the low selling price is consistent with an older vehicle, accumulating 500,000 km on a scooter over 10 years would require roughly 50,000 km per year, which is unusually high for a private two-wheeler. This suggests the mileage is likely a data entry error rather than a genuine observation. Therefore, the outlier was removed from the dataset to avoid undue influence on predictive modelling.

In [49]:
print(df['Fuel_Type'].value_counts())
print(df[df['Fuel_Type'] == 'CNG'])


Fuel_Type
Petrol    238
Diesel     58
CNG         2
Name: count, dtype: int64
   Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
18  wagon r  2015           3.25           5.09       35500       CNG   
35      sx4  2011           2.95           7.74       49998       CNG   

   Selling_type Transmission  Owner  
18       Dealer       Manual      0  
35       Dealer       Manual      0  


### Rare Fuel Type (CNG)

The dataset contains only 2 CNG vehicles, compared with 238 Petrol and 58 Diesel vehicles. Although CNG is a very rare category, the observations appear to be valid and do not indicate data quality issues. Therefore, the CNG category was retained during data cleaning to preserve the original dataset. The small sample size may limit the model's ability to learn meaningful patterns for this category, particularly if both observations fall into the same train or test split. This issue is more appropriately addressed during feature engineering or model development rather than data cleaning.

In [50]:
print('--- Final df overview ----')
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nColumns dtypes:\n{df.dtypes}')




--- Final df overview ----
Rows:    298
Columns: 9
Missing values: 0

Columns dtypes:
Car_Name             str
Year               int64
Selling_Price    float64
Present_Price    float64
Driven_kms         int64
Fuel_Type            str
Selling_type         str
Transmission         str
Owner              int64
dtype: object


In [51]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/car_clean.csv', index=False)

print('Saved to ../data/processed/car_clean.csv')
print(f'File size: {os.path.getsize('../data/processed/car_clean.csv')}')

Saved to ../data/processed/car_clean.csv
File size: 17102


## Cleaning Summary

### Duplicates
- 2 fully identical rows found (ertiga 2016, fortuner 2015) — dropped
- Dataset reduced from 301 to 299 rows

### Driven_kms Outlier
- One 2008 Activa 3G listed with 500,000 km — a second Activa 3G listing from the
  same dataset shows only 500 km, suggesting a data entry error (extra zeros) rather
  than genuine mileage. 500,000 km on a scooter implies ~50,000 km/year, far above
  typical private two-wheeler use. Row dropped.
- Dataset reduced from 299 to 298 rows

### Rare Fuel Type (CNG)
- Only 2 CNG vehicles present (vs 238 Petrol, 58 Diesel) — retained, as the
  observations are valid and not a data quality issue
- Small sample size may limit the model's ability to learn CNG-specific patterns;
  deferred to feature engineering/modelling, not a cleaning concern


### Final dataset
- 298 rows × 9 columns, 0 missing values
- Saved to `../data/processed/car_clean.csv`

### Next step
Feature engineering — create Car_Age (2018 - Year), resolve the Year/Car_Age
collinearity flagged in EDA, and decide on encoding for Fuel_Type, Selling_type,
and Transmission.